# Wan 2.1 on a free Colab T4: "You called that?" reel shots

Text-to-video with **Wan 2.1 T2V 1.3B** (Apache-2.0), tuned for Colab's free **T4 (15 GB GPU, ~12.7 GB RAM)**.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.

How it fits on a T4:
1. **Prompts first.** Wan's text encoder (UMT5-XXL) is ~11 GB, more than the free RAM can hold next to everything else.
   It is loaded straight onto the GPU, encodes every prompt, then is deleted.
2. **Then the video model.** The 1.3B transformer (fp16, ~2.8 GB) and the VAE go to the GPU; the VAE decodes in tiles.
3. **fp16 with a safety net.** The T4 has no bf16, so it runs in fp16; if a clip comes out black (NaN), that clip is
   redone in fp32 automatically.

Time on a T4: about 10–20 minutes per 5-second shot at 480×832, 25 steps. Four shots fit well inside a free session.
Edit the shots in step 2. Outputs go to `/content/shots/` and download as a zip at the end.

## 1. Install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "diffusers>=0.33" transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

## 2. Shots and settings

In [ ]:
MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
WIDTH, HEIGHT = 480, 832     # portrait 9:16 (the 1.3B model is trained at 480p; keep both multiples of 16)
FRAMES = 81                  # 5 s at 16 fps (must be 4k + 1)
STEPS = 25                   # 25 is a good T4 trade-off; 30-40 for a little more detail
GUIDANCE = 5.0
SEED = 7

LOOK = ("cinematic, photorealistic, 35mm film, shallow depth of field, soft film grain, dim living room at night lit by "
        "the flickering blue and warm glow of an unseen television, moody, high detail")
NEGATIVE = ("text, letters, words, watermark, logo, subtitles, cartoon, anime, illustration, painting, deformed hands, "
            "extra fingers, fused fingers, bad anatomy, distorted face, blurry, low quality, jpeg artifacts, "
            "overexposed, static, still image")
SHOTS = {
  "a_phone": "Close-up of a smartphone lying face up on a dark wooden coffee table, the phone screen lights up with a soft white "
             "glow, a coffee mug softly out of focus behind it, the television light flickers across the table, slow camera push-in. " + LOOK,
  "b_sofa":  "Three young friends sitting on a sofa watching a cricket match on an unseen television, leaning forward tense, then "
             "they leap up cheering with arms raised and hug in wild celebration, handheld camera, side view. " + LOOK,
  "c_hand":  "Extreme close-up, a man's hand calmly reaches into frame and picks up a glowing smartphone from a dark wooden coffee "
             "table, unhurried confident movement, television light flickering on the hand. " + LOOK,
  "d_type":  "Close-up over the shoulder of a young man holding a smartphone in both hands and typing with his thumbs, the screen "
             "glow lighting his hands, the dark room softly blurred behind, television light flickering. " + LOOK,
}

## 3. Encode the prompts (text encoder on the GPU, then freed)

In [ ]:
import gc, torch, ftfy, html, re
from transformers import AutoTokenizer, UMT5EncoderModel

def clean(t):
    t = ftfy.fix_text(t); t = html.unescape(html.unescape(t))
    return re.sub(r"\s+", " ", t).strip()

tok = AutoTokenizer.from_pretrained(MODEL, subfolder="tokenizer")
enc = UMT5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder", torch_dtype=torch.float16,
                                       device_map="cuda", low_cpu_mem_usage=True).eval()

@torch.no_grad()
def embed(text, max_len=512):    # the same as WanPipeline: trim to the real tokens, pad with zeros to 512
    t = tok([clean(text)], padding="max_length", max_length=max_len, truncation=True, add_special_tokens=True,
            return_attention_mask=True, return_tensors="pt")
    n = int(t.attention_mask.gt(0).sum())
    e = enc(t.input_ids.cuda(), t.attention_mask.cuda()).last_hidden_state[0, :n]
    return torch.cat([e, e.new_zeros(max_len - n, e.size(1))]).unsqueeze(0).cpu()

EMB = {name: embed(p) for name, p in SHOTS.items()}
NEG = embed(NEGATIVE)
del enc; gc.collect(); torch.cuda.empty_cache()
print("prompts encoded; GPU memory in use:", round(torch.cuda.memory_allocated() / 2**30, 2), "GB")

## 4. Load the video model

In [ ]:
from diffusers import AutoencoderKLWan, WanPipeline, UniPCMultistepScheduler

vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)   # 3.0 suits 480p
pipe.to("cuda")
try:
    pipe.vae.enable_tiling()
except Exception as e:
    print("VAE tiling not available:", e)
pipe.set_progress_bar_config(leave=False)
print("loaded; GPU memory in use:", round(torch.cuda.memory_allocated() / 2**30, 2), "GB")

## 5. Generate the shots

In [ ]:
import os, time, numpy as np
from diffusers.utils import export_to_video
os.makedirs("/content/shots", exist_ok=True)

def render(name, dtype):
    pipe.transformer.to(dtype)
    g = torch.Generator("cuda").manual_seed(SEED)
    out = pipe(prompt_embeds=EMB[name].cuda(), negative_prompt_embeds=NEG.cuda(), height=HEIGHT, width=WIDTH,
               num_frames=FRAMES, guidance_scale=GUIDANCE, num_inference_steps=STEPS, generator=g,
               output_type="np").frames[0]
    return out

for name in SHOTS:
    path = f"/content/shots/{name}.mp4"
    if os.path.exists(path):
        print(name, "already done"); continue
    t0 = time.time()
    frames = render(name, torch.float16)
    if not np.isfinite(frames).all() or frames.max() < 0.02:          # fp16 overflow gives NaN / black frames
        print(name, "came out black in fp16; redoing in fp32 (slower)")
        frames = render(name, torch.float32)
    pipe.transformer.to(torch.float16)
    export_to_video(list(np.nan_to_num(frames)), path, fps=16)
    print(f"{name}: {time.time() - t0:.0f} s")
    torch.cuda.empty_cache()

## 6. Upscale to 1080×1920 at 30 fps (optional)
Lanczos upscale, light sharpening and motion-blended frame interpolation with ffmpeg.

In [ ]:
import glob, subprocess
os.makedirs("/content/shots_1080", exist_ok=True)
for f in sorted(glob.glob("/content/shots/*.mp4")):
    out = f.replace("/shots/", "/shots_1080/")
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", f, "-vf",
                    "minterpolate=fps=30:mi_mode=blend,scale=1080:1920:flags=lanczos,unsharp=5:5:0.6",
                    "-c:v", "libx264", "-crf", "16", "-pix_fmt", "yuv420p", out], check=True)
    print("upscaled", out)

## 7. Watch and download

In [ ]:
from IPython.display import Video, display
for f in sorted(glob.glob("/content/shots/*.mp4")):
    print(f); display(Video(f, embed=True, width=270))

In [ ]:
!cd /content && zip -qr shots.zip shots shots_1080 2>/dev/null; ls -la /content/shots.zip
from google.colab import files
files.download("/content/shots.zip")

### If something goes wrong
- **"CUDA out of memory" in step 3:** Runtime → Restart session, then Run all again (the GPU must be empty at the start).
- **The session crashed for RAM:** you're on a CPU runtime or an older GPU; check step 1 shows a Tesla T4.
- **Slow:** lower `STEPS` to 20, or `FRAMES` to 49 (3 s).
- **Faces or hands look odd:** change `SEED` and run step 5 again; only that cell needs re-running.
- **Licence:** Wan 2.1 is Apache-2.0, fine for commercial use. Don't prompt for real people, team kits or logos.